In [1]:
import cv2
import numpy as np
import mediapipe as mp
import time
import math

C:\Users\ANURAG\AppData\Roaming\Python\Python311\site-packages\keras\src\export\tf2onnx_lib.py:8: FutureWarning: In the future `np.object` will be defined as the corresponding NumPy scalar.
  if not hasattr(np, "object"):


In [2]:
mp_face = mp.solutions.face_mesh
mp_hands = mp.solutions.hands

face_mesh = mp_face.FaceMesh()
hands = mp_hands.Hands(max_num_hands=2)

In [3]:
video = cv2.VideoCapture(0)

blink_counter = 0
hand_move_counter = 0
face_touch_counter = 0

prev_hand_x = None
start_time = time.time()

stress_percent = 0
lie_status = "Low"
emotion = "Neutral"

def distance(p1, p2):
    return math.sqrt((p1.x - p2.x)**2 + (p1.y - p2.y)**2)

while True:

    success, frame = video.read()
    if not success:
        break

    frame = cv2.flip(frame,1)
    rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)

    face_result = face_mesh.process(rgb)
    hand_result = hands.process(rgb)

    nose = None
    chin = None

    # FACE ANALYSIS 
    if face_result.multi_face_landmarks:
        for face_landmarks in face_result.multi_face_landmarks:

            # blink detection
            left_eye = face_landmarks.landmark[159].y
            right_eye = face_landmarks.landmark[386].y

            if abs(left_eye - right_eye) < 0.008:
                blink_counter += 1

            # nose & chin
            nose = face_landmarks.landmark[1]
            chin = face_landmarks.landmark[152]

            # eye contact (face center)
            if nose.x < 0.35 or nose.x > 0.65:
                eye_contact = False
            else:
                eye_contact = True
        

            # EMOTION DETECTION 
            mouth_left = face_landmarks.landmark[61]
            mouth_right = face_landmarks.landmark[291]
            mouth_top = face_landmarks.landmark[13]
            mouth_bottom = face_landmarks.landmark[14]

            mouth_width = distance(mouth_left, mouth_right)
            mouth_height = distance(mouth_top, mouth_bottom)

            if mouth_height > 0.03:
                emotion = "Happy"
            elif mouth_height < 0.015:
                emotion = "Sad"
            else:
                emotion = "Neutral"

    # HAND ANALYSIS 
    if hand_result.multi_hand_landmarks:
        for hand_landmarks in hand_result.multi_hand_landmarks:

            hand_x = hand_landmarks.landmark[9].x
            finger_tip = hand_landmarks.landmark[8]

            # hand movement
            if prev_hand_x is not None:
                movement = abs(hand_x - prev_hand_x)

                if movement > 0.03:
                    hand_move_counter += 1

            prev_hand_x = hand_x

            # FACE TOUCH DETECTION 
            if nose is not None and chin is not None:

                if distance(finger_tip, nose) < 0.05 or distance(finger_tip, chin) < 0.05:
                    face_touch_counter += 1

    # ANALYSIS EVERY 5 SECONDS
    if time.time() - start_time > 5:

        stress_score = 0

        if blink_counter > 20:
            stress_score += 1
        
        if not eye_contact:
                    stress_score += 1

        if hand_move_counter > 15:
            stress_score += 1

        if face_touch_counter > 5:
            stress_score += 1


        if stress_score == 0:
            stress_percent = 20
        elif stress_score == 1:
            stress_percent = 50
        else:
            stress_percent = 80


        if stress_percent < 30:
            lie_status = "Low"
        elif stress_percent < 60:
            lie_status = "Medium"
        else:
            lie_status = "High"


        blink_counter = 0
        hand_move_counter = 0
        face_touch_counter = 0
        prev_hand_x = None
        start_time = time.time()

    #  DISPLAY
    cv2.putText(frame,f"Blink Count: {blink_counter}",(30,40), cv2.FONT_HERSHEY_SIMPLEX,0.8,(0,255,0),2)
    cv2.putText(frame,f"Eye Contact: {eye_contact}",(30,280), cv2.FONT_HERSHEY_SIMPLEX,0.8,(255,255,255),2)
    cv2.putText(frame,f"Hand Movement: {hand_move_counter}",(30,80), cv2.FONT_HERSHEY_SIMPLEX,0.8,(255,255,0),2)
    cv2.putText(frame,f"Face Touch: {face_touch_counter}",(30,120), cv2.FONT_HERSHEY_SIMPLEX,0.8,(0,255,255),2)
    cv2.putText(frame,f"Emotion: {emotion}",(30,160), cv2.FONT_HERSHEY_SIMPLEX,0.8,(255,0,255),2)
    cv2.putText(frame,f"Stress: {stress_percent}%",(30,200), cv2.FONT_HERSHEY_SIMPLEX,0.8,(0,255,255),2)
    cv2.putText(frame,f"Lie Possibility: {lie_status}",(30,240), cv2.FONT_HERSHEY_SIMPLEX,0.8,(0,0,255),2)

    cv2.imshow("Lie Detection System",frame)

    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

video.release()
cv2.destroyAllWindows()

C:\Users\ANURAG\AppData\Roaming\Python\Python311\site-packages\google\protobuf\symbol_database.py:55: UserWarning: SymbolDatabase.GetPrototype() is deprecated. Please use message_factory.GetMessageClass() instead. SymbolDatabase.GetPrototype() will be removed soon.
  warnings.warn('SymbolDatabase.GetPrototype() is deprecated. Please '
